# Phase 7 — THE FINAL EXAM (all arms, one shot, frozen protocol)

**Runtime: L4 — MANDATORY** (the notebook enforces it; frozen protocol pins the
held-out benchmark to bf16/L4). ~2.5–3 h total, ~12–15 units. Checkpointed per
run — safe to stop and resume; finished runs are skipped.

**What it does:** evaluates the DPO and GRPO arms (3 seeds each) on the 164
held-out HumanEvalFix problems under the frozen protocol — the exact settings
of the Phase-1 baseline and the Phase-3 SFT milestone. SFT's exam rows already
exist (notebook 06), so the final cell assembles the complete headline table:
**base vs SFT vs SFT+DPO vs SFT+GRPO, 3 seeds each, vs OctoCoder-16B's 30.4%.**

Merging chain per model: rebuild merged SFT weights at `/content/merged_s{seed}`
(the path the RL adapters reference), apply the arm's adapter, merge to a final
16-bit model, hand it to the pinned harness.

In [ ]:
# 1) GPU check — frozen protocol requires bf16 on L4
!nvidia-smi -L
import torch
assert torch.cuda.is_bf16_supported(), 'Switch runtime to L4.'
name = torch.cuda.get_device_name(0)
assert 'L4' in name, f'Protocol pins the exam to L4; this is {name}. Switch runtimes.'
print(name, '- OK')

In [ ]:
# 2) Drive + what still needs running
from google.colab import drive
drive.mount('/content/drive')
import os, json, gc
PHASE3 = '/content/drive/MyDrive/rl-post-training/phase3'
PHASE4 = '/content/drive/MyDrive/rl-post-training/phase4'
PHASE5 = '/content/drive/MyDrive/rl-post-training/phase5'
PHASE7 = '/content/drive/MyDrive/rl-post-training/phase7'
os.makedirs(PHASE7, exist_ok=True)
SEEDS = [3407, 42, 1234]
RUNS = [(arm, s) for arm in ('dpo', 'grpo') for s in SEEDS]
RUNS.append(('grporand', 3407))   # the random-reward control model (attribution on held-out)
def met_path(arm, seed):
    return f'{PHASE7}/metrics_{arm}_s{seed}.json'
todo = [(a, s) for a, s in RUNS if not os.path.exists(met_path(a, s))]
print('already measured:', [(a, s) for a, s in RUNS if (a, s) not in todo])
print('to run:', todo)

In [ ]:
%%capture
!pip install unsloth
!pip uninstall -y -q torchao torchaudio torchvision timm

In [ ]:
# 3) Merge chain: SFT merge -> apply arm adapter -> final 16-bit model
from unsloth import FastLanguageModel

def ensure_merged_sft(seed):
    p = f'/content/merged_s{seed}'
    if not os.path.exists(p):
        m, t = FastLanguageModel.from_pretrained(
            f'{PHASE3}/sft_notrace_s{seed}_ep1', max_seq_length=1536,
            load_in_4bit=False, dtype=torch.bfloat16)
        m.save_pretrained_merged(p, t, save_method='merged_16bit')
        del m, t; gc.collect(); torch.cuda.empty_cache()
    return p

ADAPTER = {'dpo': lambda s: f'{PHASE4}/dpo_s{s}_ep1',
           'grpo': lambda s: f'{PHASE5}/grpo_s{s}',
           'grporand': lambda s: f'{PHASE5}/grpo_random_s{s}'}

def ensure_final(arm, seed):
    final = f'/content/final_{arm}_s{seed}'
    if os.path.exists(final):
        return final
    ensure_merged_sft(seed)   # the base path the adapter references
    m, t = FastLanguageModel.from_pretrained(
        ADAPTER[arm](seed), max_seq_length=1536,
        load_in_4bit=False, dtype=torch.bfloat16)
    m.save_pretrained_merged(final, t, save_method='merged_16bit')
    del m, t; gc.collect(); torch.cuda.empty_cache()
    print('merged', arm, seed, '->', final)
    return final

for arm, seed in todo:
    ensure_final(arm, seed)
print('all needed models merged')

In [ ]:
# 4) Harness AT THE FROZEN COMMIT
FROZEN_COMMIT = '8fc5bae6479c4fbbb28c3f8b644f6a15b3f3b5bd'
%cd /content
!rm -rf bigcode-evaluation-harness
!git clone -q https://github.com/bigcode-project/bigcode-evaluation-harness.git
%cd /content/bigcode-evaluation-harness
!git checkout -q {FROZEN_COMMIT}
ver = !git rev-parse HEAD
assert ver[0] == FROZEN_COMMIT, 'checkout failed'
print('harness pinned at', ver[0][:8])
!pip install -q -e .
!pip install -q -U transformers accelerate
!pip uninstall -y -q torchao torchaudio torchvision timm
import transformers
print('transformers version (recorded per S2.11):', transformers.__version__)

In [ ]:
# 5) THE RUNS — frozen protocol, one per (arm, seed), results to Drive
TASK = 'humanevalfixtests-python'
PROMPT = 'instruct'
BATCH = 16
%cd /content/bigcode-evaluation-harness
for arm, seed in todo:
    print('=' * 70)
    print(f'{arm.upper()} seed {seed}: 164 x 20 (~15-25 min)')
    gen_path = f'{PHASE7}/gens_{arm}_s{seed}.json'
    m_path = met_path(arm, seed)
    model_dir = f'/content/final_{arm}_s{seed}'
    !accelerate launch main.py \
      --model {model_dir} --tasks {TASK} --prompt {PROMPT} \
      --do_sample True --temperature 0.2 --top_p 0.95 --n_samples 20 \
      --batch_size {BATCH} --precision bf16 \
      --max_length_generation 2048 --allow_code_execution \
      --save_generations --save_generations_path {gen_path} \
      --metric_output_path {m_path}
    print(open(m_path).read())

In [ ]:
# 6) THE HEADLINE TABLE — the whole study on one screen
import statistics as st
BASE = {'pass@1': 17.59, 'pass@10': 23.50}
TARGET = 30.4

def read_metrics(path):
    m = json.load(open(path))
    k = [x for x in m if x != 'config'][0]
    return m[k].get('pass@1', 0) * 100, m[k].get('pass@10', 0) * 100

arms = {'sft': [f'{PHASE3}/metrics_sft_s{s}.json' for s in SEEDS],
        'dpo': [met_path('dpo', s) for s in SEEDS],
        'grpo': [met_path('grpo', s) for s in SEEDS]}
print(f"{'model':<16} pass@1    pass@10")
print(f"{'base':<16} {BASE['pass@1']:6.2f}%   {BASE['pass@10']:6.2f}%")
summary = {}
for arm, paths in arms.items():
    vals = []
    for s, p in zip(SEEDS, paths):
        if not os.path.exists(p):
            print(f'{arm} s{s}: not run'); continue
        p1, p10 = read_metrics(p)
        vals.append((p1, p10))
        print(f"{arm + ' s' + str(s):<16} {p1:6.2f}%   {p10:6.2f}%")
    if len(vals) == 3:
        m1 = st.mean(v[0] for v in vals); s1 = st.stdev([v[0] for v in vals])
        m10 = st.mean(v[1] for v in vals)
        summary[arm] = (m1, s1, m10)
if os.path.exists(met_path('grporand', 3407)):
    p1, p10 = read_metrics(met_path('grporand', 3407))
    print(f"{'grpo-RANDOM s3407':<16} {p1:6.2f}%   {p10:6.2f}%   <- the control (attribution)")
print()
for arm, (m1, s1, m10) in summary.items():
    print(f'{arm.upper():<6} mean pass@1 {m1:.2f}% (sd {s1:.2f})   mean pass@10 {m10:.2f}%   vs target {m1 - TARGET:+.2f}')
print('\nPer-seed paired deltas (arm - sft, pass@1):')
if all(len([p for p in arms[a] if os.path.exists(p)]) == 3 for a in arms):
    sft_vals = [read_metrics(p)[0] for p in arms['sft']]
    for arm in ('dpo', 'grpo'):
        deltas = [read_metrics(p)[0] - b for p, b in zip(arms[arm], sft_vals)]
        print(f"  {arm}: {'  '.join(f'{d:+.2f}' for d in deltas)}")
print('\nControl reading (dev showed RANDOM == real GRPO): if that repeats here,')
print('the honest claim is "RL process effects, not reward signal, at this budget."')
print('\nBring the WHOLE printout back — this is the study result.')

## Bring back to the session
The **entire final-table printout**. That's it — the study's headline result.

Remaining after this: significance tests on the saved generations (local, free),
the Llama cross-family rerun (units permitting), and the write-up.